In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot  as plt
import seaborn as sns
%matplotlib inline

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 5GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Data Exploration

**Loading Dataset**

In [ ]:
dataset = pd.read_csv('../input/housedata/data.csv')
dataset.info()

In [ ]:
dataset.shape

In [ ]:
dataset.head()

**Delete date column** Date column is irrelevant

In [ ]:
dataset.drop(['date'], axis = 1, inplace = True)
dataset.head()

**Checking how many different Countries are there**

In [ ]:
dataset.country.value_counts()

Deleting the country column as all the records have the same country, hence irrelevant

In [ ]:
dataset.drop(['country'], axis = 1, inplace = True)
dataset.head()

Since we already have statezip, we can safely delete street and city.

In [ ]:
dataset.drop(['street', 'city'], axis = 1, inplace = True)
dataset.head()

**Checking for null values**

In [ ]:
dataset.isnull().sum()

*No null values present*

# General corellation analysis

In [ ]:
a4_dims = (10, 8)
fig, ax = plt.subplots(figsize=a4_dims)
cor = dataset.corr()
sns.heatmap(cor, annot = True, cmap="YlGnBu")

# Analysis on number of bedroom feature

corellation of price with no. of bedrooms

In [ ]:
a4_dims = (15, 5)
fig, ax = plt.subplots(figsize=a4_dims)
sns.barplot(x = dataset.bedrooms, y = dataset.price)

*0 & 9 bedrooms are probably an outlier. Let's dive deeper*

Let's get the count  of respective no. of bedrooms

In [ ]:
dataset.groupby('bedrooms').price.agg([len, min, max])

*Hence proved that 0 & 9 are outliers. Let's remove them*

In [ ]:
df = dataset[(dataset.bedrooms > 0) & (dataset.bedrooms < 9)].copy()

In [ ]:
df.shape

# Analysis on the zipcode feature

Checking for unique zip code

In [ ]:
df.statezip.value_counts()

*All the zip codes are of Washington. Let's do a correlation analysis of zip codes*

In [ ]:
a4_dims = (5, 18)
fig, ax = plt.subplots(figsize=a4_dims)
sns.barplot(ax = ax, x = df.price, y = df.statezip)

Let's look at the distribution of price

In [ ]:
a4_dims = (15, 8)
fig, ax = plt.subplots(figsize=a4_dims)
sns.distplot(a = df.price, bins = 1000, color = 'r', ax = ax)

Groupby on price

In [ ]:
df.price.agg([min, max])

**How many instances are there with price = 0?**

In [ ]:
len(df[(df.price == 0)])

*need to set some price for these records*

# Analysis on bathroom feature w.r.t. price

In [ ]:
a4_dims = (15, 5)
fig, ax = plt.subplots(figsize=a4_dims)
sns.barplot(x = df.bathrooms, y = df.price)

# Analysis on all the instances whose price is 0

Getting all those instances

In [ ]:
zero_price = df[(df.price == 0)].copy()
zero_price.shape

In [ ]:
zero_price.head()

Let's get the unique value of the most important features

In [ ]:
sns.distplot(zero_price.sqft_living)

*Most of the 0 price houses are in the range 1000 - 5000 sqft*

Let's find more correlation between the 0 price houses

In [ ]:
zero_price.agg([min, max, 'mean', 'median'])

**We are going to use common ranges from the above table to get similar records from the original dataset and non-zero price to set the values of 0 price instances**

In [ ]:
sim_from_ori = df[(df.bedrooms == 4) & (df.bathrooms > 1) & (df.bathrooms < 4) & (df.sqft_living > 2500) & (df.sqft_living < 3000) & (df.floors < 3) & (df.yr_built < 1970)].copy()

In [ ]:
sim_from_ori.shape

In [ ]:
sim_from_ori.head()

Get the average price of these instances

In [ ]:
sim_from_ori.price.mean()

Let's confirm this by comparing with the other house price of the same yr_built and having similar sq_ft

In [ ]:
yr_sqft = df[(df.sqft_living > 2499) & (df.sqft_living < 2900)].copy()
yr_price_avg = yr_sqft.groupby('yr_built').price.agg('mean')

In [ ]:
plt.plot(yr_price_avg)

*This confirms our assumption. The avg. pricing of such houses is between 600000 to 800000*

**Replacing all 0 price values with $730000**

In [ ]:
df.price.replace(to_replace = 0, value = 735000, inplace = True)
len(df[(df.price == 0)])

In [ ]:
df.head()

# Feature reduction

Since sqft_living is the most important feature and sqft_living & sqft_above are highly corellated we are going  to remove the sqft_above feature.

In [ ]:
df.drop(['sqft_above'], axis = 1, inplace = True)
df.shape

# Handling the index order
By removing some rows our original dataset index is changed. Let's fix it

In [ ]:
df = df.reset_index()
df.info()

# Handling categorical statezip feature

Performing label encoder

In [ ]:
from sklearn import preprocessing
le = preprocessing.LabelEncoder()

In [ ]:
df['statezip_encoded'] = le.fit_transform(df.statezip)
df.head()

Let's confirm our label encoding

In [ ]:
df.statezip_encoded.value_counts()

Drop the statezip field

In [ ]:
df.drop(['statezip'], axis = 1, inplace = True)
df.head()

**One hot encoding**

In [ ]:
from sklearn.preprocessing import OneHotEncoder
ohc = OneHotEncoder()

In [ ]:
ohc_df = pd.DataFrame(ohc.fit_transform(df[['statezip_encoded']]).toarray())
# ohc_df = ohc_df.astype(int)
ohc_df.head()

Mergeing ohc_df into the main dataset

In [ ]:
df = df.join(ohc_df)
df.head()

In [ ]:
df.tail()

Drop the statezip_encoded field

In [ ]:
df.drop(['statezip_encoded'], axis = 1, inplace = True)

In [ ]:
df.info

# Splitting into train and test set

In [ ]:
df.shape

In [ ]:
X = df.iloc[:, 1:]
X.shape

In [ ]:
y = df.price

Splitting dataset into train and remainder

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_rem, y_train, y_rem = train_test_split(X, y, test_size=0.1, random_state=42)

In [ ]:
print(len(X_train) / len(df))

Splitting remainder into validation and test set

In [ ]:
X_val, X_test, y_val, y_test = train_test_split(X_rem, y_rem, test_size=0.5, random_state=42)
print(len(X_test) / len(y_rem))

Let's print the length of all the 3 splits

In [ ]:
print(len(X_train))
print(len(X_val))
print(len(X_val))

# Linear regression

In [ ]:
from sklearn.linear_model import LinearRegression
lin_reg = LinearRegression()

Fitting the model

In [ ]:
lin_reg.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import mean_squared_error
y_pred = lin_reg.predict(X_val)
mse = mean_squared_error(y_pred, y_val)
rmse = np.sqrt(mse)
rmse

In [ ]:
y_val.head(10)

In [ ]:
y_pred

In [ ]:
y_pred_test = lin_reg.predict(X_test)
mse = mean_squared_error(y_pred_test, y_test)
rmse = np.sqrt(mse)
rmse

In [ ]:
lin_reg.score(X_test, y_test)

In [ ]:
y_test

In [ ]:
y_pred_test

# Decision tree regression

In [ ]:
from sklearn.tree import DecisionTreeRegressor

reg = DecisionTreeRegressor(random_state = 42, max_depth = 10)

In [ ]:
reg.fit(X_train, y_train)

Predict

In [ ]:
reg.score(X_test, y_test)

In [ ]:
y_val.head(10)

In [ ]:
y_pred_dt